# Domain 5: Context Management & Reliability
## Task Statement 5.4: Manage Context in Codebase Exploration & Calibrate Human Review
**Certification Weight**: 17% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to maintain state across context limits using **persistent scratchpads** and **crash recovery manifests**, isolate verbose codebase scans, and design **calibrated human review** with **stratified sampling**.

---

## 🏗️ Architecture: Persistent Scratchpad & Stratified Review Pipeline

```mermaid
flowchart TD
    subgraph ContextPreservation["Context Preservation Pipeline"]
        AgentTask["Large Multi-File Refactor"] --> ScratchpadFile["Persistent Scratchpad File<br>(scratchpad.md / state_manifest.json)"]
        
        AgentTask --> ExploreWorker["Explore Subagent<br>(context: fork)"]
        ExploreWorker --> AstScan["Grep / Ast / Type Scans"]
        AstScan --> SummaryOutput["Compact Finding Summary"]
        
        SummaryOutput --> UpdateScratch["Update Scratchpad State Manifest"]
        UpdateScratch --> ScratchpadFile
    end

    subgraph ReviewCalibration["Confidence Calibration & Review"]
        DocPayload["Extracted Extraction Records"] --> ConfidenceEngine{"Field-Level Confidence<br>vs Calibrated Threshold"}
        
        ConfidenceEngine -->|"Score < 0.85"| HumanQueue["Route to Human Specialist"]
        ConfidenceEngine -->|"Score >= 0.85"| StratifiedCheck{"Stratified Random Sampling<br>(e.g. 5% audit across all categories)"}
        
        StratifiedCheck -->|"Selected for Audit"| AuditQueue["Human QA Audit Queue"]
        StratifiedCheck -->|"Bypassed"| AutoAccept["Direct Automated Acceptance"]
    end
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Exploration / Calibration Concept | Correct Implementation | Exam Trap to Reject |
|---|---|---|
| **State Manifests Across Restarts** | Save active task progress, completed milestones, and pending work to a **persistent disk file** (`manifest.json` / `scratchpad.md`). | **TRAP**: Relying entirely on in-memory variables or conversational history that vanish on session timeout or crash. |
| **Exploration Isolation** | Delegate wide codebase file searches to an **Explore subagent** with `context: fork`. | **TRAP**: Ingesting entire 50-file directory trees into the main conversation prompt. |
| **Aggregate Accuracy Pitfall** | **Calibrate by document segment/type**: A system with 98% overall accuracy may have a 60% error rate on rare edge-case forms (e.g. handwritten customs forms). | **TRAP**: Relying solely on overall aggregate accuracy metrics to justify skipping human review on high-risk document classes. |
| **Ongoing Quality Auditing** | **Stratified Random Sampling**: Sample a guaranteed percentage of high-confidence documents from *every* category to detect silent data drift. | **TRAP**: Auditing only low-confidence documents, leaving model blind spots and silent hallucinated false positives undetected. |

---

## 💡 Architectural Decision Matrix

| Mechanism | Purpose | Failure Mode Prevented |
|---|---|---|
| **`scratchpad.md`** | Preserves hypotheses, notes, and task progress | Context compaction and catastrophic forgetting |
| **Explore Subagent (`context: fork`)** | Gathers search results and provides synthesis | Main context pollution from massive grep outputs |
| **Stratified Sampling** | Ongoing human evaluation across categories | Undetected accuracy drops in rare document classes |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import random
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 5.4 Scratchpads & Human Review Routing.")

### 📝 Step 2: Scratchpad & Human Review Routing Engine
We implement the working memory scratchpad and confidence router with stratified sampling.

In [ ]:
# Step 2: Scratchpad & Router Logic
@dataclass
class ExtractionRecord:
    doc_id: str
    category: str
    confidence: float
    extracted_amount: float
    requires_human_review: bool = False


class HumanReviewRoutingEngine:
    def __init__(self, confidence_threshold: float = 0.85) -> None:
        self.threshold = confidence_threshold

    def route_record(self, record: ExtractionRecord) -> ExtractionRecord:
        record.requires_human_review = (record.confidence < self.threshold)
        return record

    @staticmethod
    def stratified_random_sample(records: List[ExtractionRecord], sample_per_stratum: int = 1) -> List[ExtractionRecord]:
        """Selects representative audit samples across distinct categories/strata."""
        strata = defaultdict(list)
        for r in records:
            strata[r.category].append(r)
        
        sampled = []
        for cat, items in strata.items():
            sampled.extend(items[:sample_per_stratum])
        return sampled


class CodebaseScratchpadManager:
    """External working memory container preventing conversation context bloat."""
    def __init__(self) -> None:
        self.findings: List[str] = []

    def record_finding(self, observation: str) -> None:
        self.findings.append(observation)

    def render_scratchpad(self) -> str:
        return "### AGENT WORKING SCRATCHPAD\n" + "\n".join(f"- {f}" for f in self.findings)

print(" HumanReviewRoutingEngine and CodebaseScratchpadManager ready.")

### 🧪 Step 3: Interactive Simulation
1. Route high vs low confidence records.
2. Execute stratified audit sampling across document segments.
3. Record exploration observations into scratchpad.

In [ ]:
# Step 3: Simulation
router = HumanReviewRoutingEngine(confidence_threshold=0.85)

rec_high = router.route_record(ExtractionRecord("D1", "standard_invoice", 0.96, 120.0))
rec_low = router.route_record(ExtractionRecord("D2", "customs_declaration", 0.72, 450.0))

print("High Confidence (0.96) Needs Human:", rec_high.requires_human_review)
print("Low Confidence (0.72) Needs Human:", rec_low.requires_human_review)

# Stratified audit across categories
records = [
    ExtractionRecord("D10", "Category_A", 0.95, 10.0),
    ExtractionRecord("D11", "Category_A", 0.92, 20.0),
    ExtractionRecord("D20", "Category_B", 0.88, 30.0),
]
audit_sample = HumanReviewRoutingEngine.stratified_random_sample(records, sample_per_stratum=1)
print(f"\nStratified Sample Count: {len(audit_sample)} (1 from Category_A, 1 from Category_B)")

# Scratchpad memory test
pad = CodebaseScratchpadManager()
pad.record_finding("auth_service uses bcrypt with 12 rounds.")
pad.record_finding("redis cache TTL is set to 3600 seconds.")
print("\n", pad.render_scratchpad())

### 🔬 Step 4: Verification Assertions
We assert that:
1. High-confidence item bypasses human review.
2. Low-confidence item is routed to human review.
3. Stratified sampling captures 1 sample per distinct stratum.
4. Scratchpad renders clean, formatted findings.

In [ ]:
# Step 4: Verification Checks
assert rec_high.requires_human_review is False
assert rec_low.requires_human_review is True

assert len(audit_sample) == 2
strata_represented = {r.category for r in audit_sample}
assert "Category_A" in strata_represented
assert "Category_B" in strata_represented

scratchpad_text = pad.render_scratchpad()
assert "bcrypt" in scratchpad_text
assert "redis cache" in scratchpad_text

print(" All verification assertions passed! Scratchpads & review routing verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Maintain an **external scratchpad** for working memory during multi-step codebase exploration.
- **Rule 2**: Route items below the confidence threshold (<0.85) to human specialists.
- **Rule 3**: Apply **stratified random sampling** across document classes to audit high-confidence decisions for silent model drift.